In [1]:
import os


In [2]:
%pwd

'c:\\Users\\belal\\Text-Summarizer-Project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\belal\\Text-Summarizer-Project'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    ALL_REQUIRED_FILES: list

In [6]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories


In [12]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath: Path = CONFIG_FILE_PATH,
            params_filepath: Path = PARAMS_FILE_PATH):
        
            self.config = read_yaml(config_filepath)
            self.params = read_yaml(params_filepath)

            create_directories([self.config.artifacts_root])


    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=Path(config.root_dir),
            STATUS_FILE=config.STATUS_FILE,
            ALL_REQUIRED_FILES=config.ALL_REQUIRED_FILES
        )

        return data_validation_config



In [13]:
import os
from textSummarizer.logging import logger


In [14]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_files(self) -> bool:
        try:
            dataset_dir = os.path.join("artifacts", "data_ingestion", "samsum_dataset")
            all_files = os.listdir(dataset_dir)

            required_files = set(self.config.ALL_REQUIRED_FILES)
            present_files = set(all_files)

            
            missing_files = required_files - present_files

            
            extra_files = present_files - required_files

            validate_status = len(missing_files) == 0 and len(extra_files) == 0

            with open(self.config.STATUS_FILE, "w") as f:
                f.write(f"Data Validation Status: {validate_status}\n")
                f.write(f"Missing Files: {list(missing_files)}\n")
                f.write(f"Extra Files: {list(extra_files)}\n")

            return validate_status

        except Exception as e:
            raise e

In [15]:
try:
    config_manager = ConfigurationManager()
    data_validation_config = config_manager.get_data_validation_config()

    data_validation = DataValidation(config=data_validation_config)
    validation_status = data_validation.validate_all_files()

except Exception as e:
    raise e

[2026-04-28 07:59:02,975: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-04-28 07:59:02,977: INFO: common]: yaml file: params.yaml loaded successfully
[2026-04-28 07:59:02,979: INFO: common]: created directory at: artifacts
[2026-04-28 07:59:02,982: INFO: common]: created directory at: artifacts/data_validation
